In [1]:
import mlflow 
import pandas as pd 
from pprint import pprint
from pytorch_pipeline.utils import resolve_uri
from pytorch_pipeline.train.factory import set_device


/home/etienne/miniconda3/envs/inat_cv/lib/python3.10/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /opt/conda/conda-bld/pytorch_1729647382455/work/c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0
/home/etienne/miniconda3/envs/inat_cv/lib/python3.10/site-packages/torch/amp/grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


In [2]:
model_name = 'test'
model_version = 13

In [3]:
mlflow.set_tracking_uri(resolve_uri())
device = set_device('cuda')

# Construct the model URI
model_uri = f"models:/{model_name}/{model_version}"

# Load the model
model = mlflow.pyfunc.load_model(model_uri)


Running on cuda


2026/09/16 12:37:28 INFO mlflow.models.signature: Unsupported type hint: pd.DataFrame, skipping schema inference
/home/etienne/miniconda3/envs/inat_cv/lib/python3.10/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [4]:
pprint(model._model_impl.python_model.model_params)
pprint(model._model_impl.python_model.class_thresholds)

ModelParams(backbone='bioclip',
            head_neurons=256,
            head_outputs=1,
            head_dropout_prob=0.5,
            attention_neurons=128,
            attention_dropout_prob=0.0,
            start_unfreezed=1,
            gated=True)
array([0.65, 0.77, 0.7 ])


In [5]:
paths = []
imgs =[
    [161284642,161284591],
    [147565223]
]
for b in imgs:
    paths.append([f"/home/etienne/projects/inat-phenology-cv/data/images/{i}.jpg" for i in b])

pprint(paths)

[['/home/etienne/projects/inat-phenology-cv/data/images/161284642.jpg',
  '/home/etienne/projects/inat-phenology-cv/data/images/161284591.jpg'],
 ['/home/etienne/projects/inat-phenology-cv/data/images/147565223.jpg']]


In [6]:
model_input = pd.DataFrame(
    {
        "observation_id" : [97028130, 89365289],
        "paths" : paths
    }
)
display(model_input)

,observation_id,paths
0,97028130,[/home/etienne/projects/inat-phenology-cv/data...
1,89365289,[/home/etienne/projects/inat-phenology-cv/data...


In [7]:
predictions, weights = model.predict(model_input)
print(predictions)
pprint(weights)

[[1 1 1]
 [1 1 0]]
[{'Flower_Budding': [0.16903579235076904, 0.830964207649231],
  'Flowering': [0.28319936990737915, 0.7168006896972656],
  'Fruiting': [0.7410027980804443, 0.2589971721172333]},
 {'Flower_Budding': [1.0], 'Flowering': [1.0], 'Fruiting': [1.0]}]
